# Lab: four models, one split

The risk team reviews clients who might default next month. One row is one credit-card client; `default` is 1 when the client defaulted. The team asks: "Which model ranks the risky clients best, and how many defaulters would it find if we review the top 10%?"

Work through the checkpoints in order. C1 is the minimum, C2 is today's target, C3 is for those who finish C2.

## The starter

The cell loads the data and splits it 80 / 20 with `stratify=y, random_state=42`, the same split as 5.2. `scores(model)` takes a fitted model and returns accuracy, ROC AUC, average precision, and precision, recall and F1 at the default 0.5 threshold, all on the test rows.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split

NAMES = (["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
         + [f"BILL_AMT{i}" for i in range(1, 7)] + [f"PAY_AMT{i}" for i in range(1, 7)])

df = pd.read_csv("data/credit_default.csv")
df = df.drop(columns=[c for c in df.columns if c.upper() == "ID"])
target = [c for c in df.columns if c.lower() in ("y", "default", "default.payment.next.month",
                                                 "default payment next month")][0]
y = df.pop(target).astype(int).rename("default")
if list(df.columns[:2]) == ["X1", "X2"]:            # the UCI download names the columns X1 ... X23
    df.columns = NAMES
X = df.rename(columns={"PAY_1": "PAY_0"})[NAMES]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)


def scores(model):
    """Test-row metrics for a fitted classifier; precision, recall and F1 at the 0.5 threshold."""
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    return {"accuracy": accuracy_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba),
            "avg_precision": average_precision_score(y_test, proba),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred, zero_division=0)}


print(len(X_train), "training clients,", len(X_test), "test clients,", y_test.sum(), "test defaulters",
      f"(base rate {y_test.mean():.3f})")

## C1 · Two baselines

### Step 1 · Score a `DummyClassifier(strategy="most_frequent")`

### Step 2 · Fit `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` and score it

### Step 3 · Fit `DecisionTreeClassifier(max_depth=5, random_state=42)` and score it

## C2 · Two ensembles

### Step 4 · Fit `RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)` and score it

It is the slowest of the four to fit.

### Step 5 · Fit `HistGradientBoostingClassifier(random_state=42)` and score it

Print `n_iter_`: early stopping is on by default when there are more than 10,000 rows.

### Step 6 · One table, sorted by ROC AUC

### Step 7 · "Which model ranks the risky clients best?"

Translate: the highest ROC AUC and average precision. Then say why the accuracy column could not have answered the question.

## C3 · The review list

### Step 8 · The 600 clients each model ranks highest

For the boosted model and the logistic regression, take the 600 test clients with the highest predicted probability (10% of 6,000) and count how many defaulted.

### Step 9 · `class_weight="balanced"`

Refit both with `class_weight="balanced"`. Print precision, recall and ROC AUC again. What changed, and what did not?

### Step 10 · Three sentences for the risk team

The model you recommend, its test ROC AUC, and how many defaulters it puts in a review list of 600.

## Bonus · A forest of shallower trees

## Check yourself